# W06-A · Word vectors from café orders

A customer searches a café app for *cold tea*. The best match is *iced tea*, but to a computer that only compares
spellings, *cold* and *iced* are two different words with nothing in common.

The idea of this notebook: **a word is known by the words around it**. *iced* and *cold* both come just before
*latte*, *tea* and *coffee*, so they must be used in the same way. You will turn that idea into numbers, twice: first
by counting neighbours, then with **word2vec**, which learns a short list of numbers for every word by playing a
guessing game. You build it by hand on 20 café orders, then train it on 30,000 real reviews.

By the end of this notebook you will be able to:
1. list every word's **neighbours**: the word just before it and the word just after it
2. turn a word's neighbour counts into a **vector**, and find the words most like it
3. write the **sigmoid**, which turns any score into a chance between 0 and 1
4. write word2vec's **nudge** step, and train word vectors with it
5. train word2vec on **30,000 real reviews**, and see what it learns: *fridge* is close to *refrigerator*
6. see that words with opposite meanings, like *hot* and *cold*, still end up close
7. learn **product vectors** from customers' product lists, for a *"you may also like"* box

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shripada/ads6703-nlp/blob/main/w06/W06-A-word-vectors.ipynb) · On Colab, run the next cell first. On your laptop, you can skip it.

In [ ]:
# Only for Google Colab: fetch the course folder (with the data) and move into this week's folder.
# On your laptop this cell does nothing.
import os, subprocess, sys
if "google.colab" in sys.modules:
    if os.path.exists("/content/ADS6703"):
        subprocess.run(["git", "-C", "/content/ADS6703", "pull", "-q"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", "https://github.com/shripada/ads6703-nlp.git", "/content/ADS6703"], check=True)
    os.chdir("/content/ADS6703/w06")

In [ ]:
# ▶ Setup: just run this cell. These 20 café orders are all the text our model will ever read.
import math                       # math.sqrt (square root) and math.exp (e to the power of x)
import random                     # random numbers, to start the word vectors
from collections import Counter   # Counter: a dictionary that counts things

orders = [
    "iced latte please",      "cold latte please",
    "iced tea with lemon",    "cold tea with lemon",
    "iced coffee with milk",  "cold coffee with milk",
    "hot tea with honey",     "hot coffee with milk",
    "hot chai with ginger",   "masala chai with ginger",
    "masala chai please",     "hot latte please",
    "hot chai please",        "masala tea with ginger",
    "lemon cake please",      "chocolate cake please",
    "carrot cake please",     "chocolate muffin please",
    "carrot cake with cream", "chocolate cake with cream",
]
words = sorted(set(" ".join(orders).split()))   # every different word, in alphabetical order
# How many orders and how many different words, then the words themselves
print(len(orders), "orders,", len(words), "different words:")
print(words)

> 📖 Look for *iced* and *cold*: they are never in the same order. A computer can still learn that they are alike,
> because they keep turning up next to the same words.

## 1 · Every word's neighbours

A word's **neighbours** are the word just before it and the word just after it. In *"iced latte please"*:

| word | its neighbours |
|---|---|
| iced | latte |
| latte | iced, please |
| please | latte |

We write each one as a pair: (word, neighbour). So *"iced latte please"* gives four pairs:
(iced, latte), (latte, iced), (latte, please), (please, latte).

### ✍️ Exercise 1: `make_pairs(order)`

Split the order into words. For each word, add a pair with the word before it (if there is one) and a pair with the
word after it (if there is one).

In [ ]:
def make_pairs(order):
    order_words = order.split()             # a list of strings, e.g. ["iced", "latte", "please"]
    pairs = []                              # a list of (word, neighbour) tuples
    for i in range(len(order_words)):       # i = the position of the word
        if i > 0:
            ...   # ✍️ add the pair (this word, the word before)
        if i < len(order_words) - 1:
            ...   # ✍️ add the pair (this word, the word after)
    return pairs
# Check: the four pairs of "iced latte please", and none for a one-word order
print(make_pairs("iced latte please"))
assert make_pairs("iced latte please") != [], "make_pairs returned an empty list: fill in the two ✍️ lines"
assert make_pairs("iced latte please") == [("iced", "latte"), ("latte", "iced"), ("latte", "please"), ("please", "latte")], "not quite: compare with the four pairs above"
assert make_pairs("tea") == [], "a one-word order has no neighbours"
print("✅")

In [ ]:
# ▶ Just run this cell: the pairs of all 20 orders, in one list of (word, neighbour) tuples.
pairs = []
for order in orders:
    pairs = pairs + make_pairs(order)   # + joins two lists into one longer list
print(len(pairs), "pairs. The first six:", pairs[:6])

## 2 · A word as a list of neighbour counts

For one word, count how many times each word of the vocabulary is its neighbour. That list of counts, one per word,
in alphabetical order, is the word's **vector**.

### ✍️ Exercise 2: `neighbour_vector(word)`

Go through all the pairs. Count the neighbours of the pairs that start with `word`. Then build a list with one count
per word of `words`.

In [ ]:
def neighbour_vector(word):
    neighbour_counts = Counter()                  # a Counter: neighbour -> how many times it was seen
    for word_in_pair, neighbour in pairs:         # each pair is a tuple of two strings
        if word_in_pair == word:
            ...   # ✍️ add 1 to the count of this neighbour
    vector = []                                   # a list of numbers: one count per vocabulary word
    for vocab_word in words:                      # always the same order: alphabetical
        ...   # ✍️ add the count of `vocab_word` to the vector
    return vector
# Check: iced and cold should get the same list of counts
print("iced:", neighbour_vector("iced"))
print("cold:", neighbour_vector("cold"))
assert neighbour_vector("iced") != [], "neighbour_vector returned an empty list: fill in the two ✍️ lines"
assert len(neighbour_vector("iced")) == len(words), "one number per word of the vocabulary"
assert sum(neighbour_vector("iced")) == 3, "iced appears 3 times, always first, so it has 3 neighbours"
assert neighbour_vector("iced")[words.index("latte")] == 1, "latte is a neighbour of iced once"
print("✅")

> 📖 **Reading the output.** The two lists are the same: *iced* and *cold* each have *latte*, *tea* and *coffee* as
> neighbours, once each. Every other number is 0.

To compare two vectors we use **cosine similarity**: the dot product divided by both lengths. 1 means the two arrows
point the same way, 0 means nothing in common. Just run the cell.

In [ ]:
# ▶ Just run this cell: cosine similarity, and a function that finds a word's closest words.
def dot(vec_a, vec_b):
    # dot product: multiply the numbers in the same position, then add them all up
    total = 0
    for num_a, num_b in zip(vec_a, vec_b):    # zip pairs them up: (vec_a[0], vec_b[0]), (vec_a[1], vec_b[1]), ...
        total = total + num_a * num_b
    return total

def length(vec):
    return math.sqrt(dot(vec, vec))           # √(sum of squares)

def cosine(vec_a, vec_b):
    if length(vec_a) == 0 or length(vec_b) == 0:
        return 0                              # a list of all 0s points nowhere: call it "nothing in common"
    return dot(vec_a, vec_b) / (length(vec_a) * length(vec_b))

def nearest(word, word_to_vector, top=3):
    # word_to_vector is a dictionary: word (a string) -> its vector (a list of numbers)
    cosine_and_word = []                      # a list of (cosine, word) tuples
    for other_word in word_to_vector:         # looping over a dictionary gives its keys: the words
        if other_word != word:                # skip the word itself: its cosine with itself is always 1
            score = cosine(word_to_vector[word], word_to_vector[other_word])
            cosine_and_word.append((score, other_word))   # cosine first, so sorting sorts by it
    cosine_and_word.sort(reverse=True)        # biggest cosine first
    return [(other_word, round(score, 2)) for score, other_word in cosine_and_word[:top]]   # the top few, as (word, cosine)

# Every word's neighbour-count vector, in one dictionary: word -> its vector
word_to_count_vector = {}
for word in words:
    word_to_count_vector[word] = neighbour_vector(word)

# The 3 closest words for a few words. {word:10s} pads the word to 10 characters, so the lists line up
for word in ["iced", "cold", "tea", "chocolate", "cake"]:
    print(f"{word:10s}", nearest(word, word_to_count_vector))

> 📖 **Reading the output.** *iced* and *cold* have a cosine of 1, though they never appear in the same order.
> *tea* is closest to *coffee*, *chocolate* to *carrot*, *cake* to *muffin*. Counting neighbours already works.
>
> The catch: each vector has one number per word of the vocabulary, here 19, and most of them are 0. A real
> vocabulary has 50,000 words or more, so every word would get 50,000 numbers, almost all 0. Word2vec gives every word
> a **short** list instead, here only 5 numbers, and **learns** them.

## 3 · Word2vec's guessing game

Word2vec plays a guessing game: *"is this a real neighbour pair, or a fake one?"*

- **real pairs** come from the text: (iced, latte)
- **fake pairs** take a word and a random word: (iced, cake)

Every word gets a short list of numbers. The **score** of a pair is the dot product of the two lists. The
**sigmoid** turns any score into a chance between 0 and 1: the model's guess that the pair is real.

$$\sigma(x) = \frac{1}{1 + e^{-x}}$$

A score of 0 gives 0.5 (no idea). A big score gives nearly 1, a very negative score nearly 0. Python writes
$e^{-x}$ as `math.exp(-x)`.

### ✍️ Exercise 3: `sigmoid(x)`

In [ ]:
def sigmoid(x):
    return ...   # ✍️ 1 divided by (1 + e to the power of -x)
assert sigmoid(0) is not ..., "sigmoid returned ...: fill in the ✍️ line"
# Check: the chance for a few scores, from very negative to very positive
for x in [-2, -1, 0, 1, 2]:
    print(f"sigmoid({x:2d}) = {sigmoid(x):.2f}")
assert sigmoid(0) == 0.5, "a score of 0 means no idea: 0.5"
assert round(sigmoid(1), 2) == 0.73 and round(sigmoid(-1), 2) == 0.27, "not quite"
print("✅")

## 4 · The nudge

After each guess, word2vec **nudges** the two lists a little, so the next guess is better.

- the **target** is 1 for a real pair, 0 for a fake one
- the **error** is target − guess: positive for a real pair, negative for a fake one
- each list moves by **rate × error × the other list**

$$w_{\text{new}} = w + \text{rate} \times \text{error} \times c \qquad c_{\text{new}} = c + \text{rate} \times \text{error} \times w$$

A real pair pulls the two lists towards each other, a fake pair pushes them apart.

The **rate** (also called the **learning rate**) is a small number you choose, such as 0.05. It sets how big each
step is. Too big, and the lists jump around with every pair; too small, and training needs many more rounds. Small
steps, repeated many times, let the lists settle where they suit all the pairs.

One detail: every word has **two** lists. *w* is its list when it is the word in the middle, *c* when it is the
neighbour. In the code they are called `word_vec` and `neighbour_vec`. Both lists change together, so both new values
must use the **old** numbers.

### ✍️ Exercise 4: `nudge(word_vec, neighbour_vec, target, rate)`

Work out the guess and the error, then change both lists, number by number. The lists are changed in place, so the
function returns nothing.

In [ ]:
def nudge(word_vec, neighbour_vec, target, rate):
    # word_vec and neighbour_vec are lists of numbers (w and c in the formula)
    guess = ...   # ✍️ the sigmoid of the dot product of word_vec and neighbour_vec
    error = ...   # ✍️ target minus guess
    for k in range(len(word_vec)):
        old_w, old_c = word_vec[k], neighbour_vec[k]   # keep the old numbers: both updates use them
        word_vec[k] = ...        # ✍️ old_w + rate × error × old_c
        neighbour_vec[k] = ...   # ✍️ old_c + rate × error × old_w
# The example from the slides: iced = [1, 0], latte = [0, 1], rate 0.5
iced, latte = [1, 0], [0, 1]
nudge(iced, latte, 1, 0.5)               # a real pair
assert ... not in iced + latte, "nudge left ... in the lists: fill in the four ✍️ lines"
print("real pair:", iced, latte, " new guess:", round(sigmoid(dot(iced, latte)), 2))
assert iced == [1, 0.25] and latte == [0.25, 1], "not quite: the guess is 0.5, the error 0.5, so each list moves by 0.25 × the other"

iced, cake = [1, 0], [0, 1]
nudge(iced, cake, 0, 0.5)                # a fake pair
print("fake pair:", iced, cake, " new guess:", round(sigmoid(dot(iced, cake)), 2))
assert iced == [1, -0.25] and cake == [-0.25, 1], "not quite: for a fake pair the error is 0 − 0.5 = −0.5"
print("✅")

> 📖 **Reading the output.** The real pair's guess goes up from 0.5 to 0.62; the fake pair's goes down to 0.38. One
> nudge is a small step. Training repeats it thousands of times.

## 5 · Train word vectors

Training starts every list with small random numbers, then goes through all the pairs 300 times. For each real pair
it nudges towards 1, and it makes up 2 fake pairs (the same word with a random word) and nudges those towards 0.
Just run the cell: it takes a few seconds.

In [ ]:
# ▶ Just run this cell.
def train(pairs, vocab, size=5, rounds=300, rate=0.05, fakes=2, seed=1):
    random.seed(seed)                     # the same random numbers every time you run it
    # Two dictionaries, word -> a list of `size` numbers:
    # word_vectors for the word in the middle, neighbour_vectors for the word as a neighbour
    word_vectors, neighbour_vectors = {}, {}
    for word in vocab:                    # each list starts as random numbers between -0.5 and 0.5
        word_vectors[word] = [random.uniform(-0.5, 0.5) for _ in range(size)]
    for word in vocab:
        neighbour_vectors[word] = [random.uniform(-0.5, 0.5) for _ in range(size)]
    pairs = list(pairs)                   # a copy, so shuffling does not change the original
    for _ in range(rounds):
        random.shuffle(pairs)             # a new order each round
        for word, neighbour in pairs:
            nudge(word_vectors[word], neighbour_vectors[neighbour], 1, rate)              # a real pair
            for _ in range(fakes):
                random_word = random.choice(vocab)
                nudge(word_vectors[word], neighbour_vectors[random_word], 0, rate)        # a fake pair
    return word_vectors, neighbour_vectors

word_vectors, neighbour_vectors = train(pairs, words)

# After training: a real pair should get a guess near 1, a pair never seen a guess near 0
print("guess for (iced, latte), a real pair:", round(sigmoid(dot(word_vectors["iced"], neighbour_vectors["latte"])), 2))
print("guess for (iced, cake), never seen:  ", round(sigmoid(dot(word_vectors["iced"], neighbour_vectors["cake"])), 2))
print()
# The 5 learned numbers of iced and cold, rounded to 2 places
print("iced:", [round(number, 2) for number in word_vectors["iced"]])
print("cold:", [round(number, 2) for number in word_vectors["cold"]])

> 📖 **Reading the output.** The model now gives a high chance to (iced, latte) and a low one to (iced, cake). The
> five numbers of *iced* and *cold* are close: both words were pulled towards the same neighbours, *latte*, *tea* and
> *coffee*, so they ended up pointing the same way. The numbers on their own mean nothing; only how they compare does.

The word vectors are the lists in the `word_vectors` dictionary. Compare them with cosine, as before. Just run the cell.

In [ ]:
# ▶ Just run this cell: the same nearest() as before, now with the 5-number word2vec vectors.
for word in ["iced", "cold", "tea", "chocolate", "cake"]:
    print(f"{word:10s}", nearest(word, word_vectors))

> 📖 **Reading the output.** Five numbers per word, instead of 19, and the same words come out close: *iced* and *cold*,
> *tea* and *coffee*, *cake* and *muffin*. On 20 orders, counting does just as well: the model has seen too little
> to show what it can do. That needs real text, in section 6.

### Try your own

Change the word and run the cell again. Try *chai*, *latte*, *lemon* or *please*.

In [ ]:
word = "chai"    # ✍️ try your own
# The same word, compared both ways: neighbour counts first, then word2vec
print("counts:  ", nearest(word, word_to_count_vector))
print("word2vec:", nearest(word, word_vectors))

## 6 · Word2vec on 30,000 real reviews

Twenty orders are enough to see how word2vec works, not to see what it can do. So now train it on real text: the
**30,000 Amazon reviews** of home appliances and parts, about 1.2 million words.

Your `train` loop would take hours on that much text. The **gensim** library runs exactly the same nudge, written in
fast C code, in a few seconds. First, split every review into sentences, and every sentence into lowercase words.
Just run the cell. If it prints *Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'*, ignore it: that
message is harmless.

In [ ]:
# ▶ Just run this cell: load the 30,000 reviews and split them into sentences of words.
import csv, gzip, re

try:
    from gensim.models import Word2Vec    # the word2vec library
except ImportError:                       # not installed yet (e.g. on Colab): install it, once
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gensim"], check=True)
    from gensim.models import Word2Vec

DATA = "../data/"   # notebooks live in ADS6703/w06/, data in ADS6703/data/
with gzip.open(DATA + "reviews.csv.gz", "rt", encoding="utf-8") as file:
    reviews = list(csv.DictReader(file))  # a list of dictionaries, one per review

sentences = []                                         # a list of sentences; each sentence is a list of words
for review in reviews:                                 # review is a dictionary: column name -> value
    # the title and the review as one text; <br /> is a line break left over from the web page
    text = (review["title"] + ". " + review["text"]).replace("<br />", " ")
    for sentence_text in re.split(r"[.!?]+", text):    # split at . ! and ?
        sentence_words = re.findall(r"[a-z]+", sentence_text.lower())   # the words, lowercase, letters only
        if len(sentence_words) > 1:                    # a one-word sentence has no neighbours: skip it
            sentences.append(sentence_words)

total_words = sum(len(sentence_words) for sentence_words in sentences)
print(len(reviews), "reviews,", len(sentences), "sentences,", total_words, "words")
print("one sentence:", sentences[1])

Now train. Each setting is something you have already met:

| setting | meaning | in your `train` |
|---|---|---|
| `sg=1` | make (word, neighbour) pairs: skip-gram | `make_pairs` |
| `window=5` | up to 5 neighbours on each side, not just 1 | one word each side |
| `negative=5` | 5 fake pairs for every real pair | `fakes=2` |
| `vector_size=100` | 100 numbers per word | `size=5` |
| `epochs=5` | go through the text 5 times | `rounds=300` |
| `min_count=5` | skip words seen fewer than 5 times | (none) |

Just run the cell: it takes about 10 seconds.

In [ ]:
# ▶ Just run this cell.
word_model = Word2Vec(sentences, sg=1, window=5, negative=5, vector_size=100, epochs=5,
                      min_count=5, workers=1, seed=1)   # workers=1 and seed=1: the same result every time you run it
review_word_vectors = word_model.wv      # works like a dictionary: review_word_vectors["fridge"] is a list of 100 numbers

print(len(review_word_vectors), "words have a vector")
# float() turns gensim's number type into a plain Python number, so it prints neatly
print("the first 8 of the 100 numbers for 'fridge':", [round(float(number), 2) for number in review_word_vectors["fridge"][:8]])

`review_word_vectors.most_similar(word)` does what your `nearest` did: it finds the words with the biggest cosine. Just run the
cell.

In [ ]:
# ▶ Just run this cell.
def closest(word, top=5):
    # most_similar gives a list of (word, cosine) tuples, biggest cosine first; we round each cosine to 2 places
    similar = review_word_vectors.most_similar(word, topn=top)
    return [(similar_word, round(float(score), 2)) for similar_word, score in similar]

for word in ["fridge", "keurig", "gift", "easy", "happy"]:
    print(f"{word:10s}", closest(word))

> 📖 **Reading the output.** No one told the model any of this. *fridge* is closest to *refrigerator*, and then to
> *frig* and *refridgerator*: it learned the short forms and misspellings customers use. *keurig* (a coffee-machine
> brand) is closest to *kuerig* and *kurig*. *gift* is close to *birthday* and *christmas*, *easy* to *simple* and
> *painless*, *happy* to *pleased* and *satisfied*. A search box with these vectors could find a review that says
> *the frig was a simple setup* when you searched for *easy fridge*. (Look at *happy* again in section 7.)

### Back to *cold* and *iced*

Just run the cell.

In [ ]:
# ▶ Just run this cell.
# similarity(a, b) is the cosine of the two words' vectors
print("cosine of cold and iced:", round(float(review_word_vectors.similarity("cold", "iced")), 2))
print("cold:", closest("cold"))

> 📖 **Reading the output.** In 30,000 reviews, *cold* and *iced* get a cosine of about 0.6, though the model was never
> told what either word means. Look at the other words close to *cold*: that is the next section.

### Try your own

Change the word and run the cell again. Try *coffee*, *espresso*, *perfect*, *beautiful*, *milk* or a brand name. A word seen
fewer than 5 times has no vector, and gives an error.

In [ ]:
word = "coffee"    # ✍️ try your own
print(closest(word, top=8))

## 7 · Opposites end up close

Just run the cell.

In [ ]:
# ▶ Just run this cell.
# The cosine of three pairs of opposites, then the 8 words closest to happy
for word_a, word_b in [("hot", "cold"), ("good", "bad"), ("happy", "unhappy")]:
    print(f"cosine of {word_a} and {word_b}:", round(float(review_word_vectors.similarity(word_a, word_b)), 2))
print("happy:", closest("happy", top=8))

> 📖 **Reading the output.** *hot* and *cold* get about 0.7, *good* and *bad* about 0.6, *happy* and *unhappy* about 0.8. Opposites are used in the
> same places (*hot water*, *cold water*; *I was happy with it*, *I was unhappy with it*), so their vectors end up close. Look at
> *happy*: *pleased* and *satisfied*, as expected, but also *unhappy* and *disappointed*. Word vectors know which words are used
> **alike**, not which ones mean the **same**. A search for *cold water* could happily return *hot water*.

## 8 · Product vectors from real customers

The same idea works for anything that comes in a sequence. Treat each **customer's list of products** as a
"sentence" and each **product** as a "word". Products reviewed by the same customers get similar vectors. This is how
a shop can fill a *"you may also like"* box without reading a single product description.

Each review has a `user_id` (the customer) and a `parent_asin` (Amazon's code for the product).

### ✍️ Exercise 5: each customer's products

Build a dictionary: customer → the list of products they reviewed, oldest first. The reviews are sorted by date for
you.

In [ ]:
by_date = sorted(reviews, key=lambda review: review["date"])   # a list of reviews, oldest first
customer_to_products = {}                                      # a dictionary: user_id -> list of product codes
for review in by_date:
    user, product = review["user_id"], review["parent_asin"]
    if user not in customer_to_products:
        ...   # ✍️ a new customer: give them an empty list
    ...   # ✍️ add this product to the customer's list
# Check: every customer appears once, and every one of the 30,000 reviews is in some list
print(len(customer_to_products), "customers")
assert customer_to_products != {}, "customer_to_products is empty: fill in the two ✍️ lines"
assert len(customer_to_products) == 10941, "not quite: there are 10,941 different customers"
assert sum(len(products) for products in customer_to_products.values()) == 30000, "every review should be in exactly one list"
print("✅")

In [ ]:
# ▶ Just run this cell: train product vectors, then a "you may also like" box for four products.
# A list of product lists, one per customer. A customer with one product has no neighbours: leave them out
product_lists = [products for products in customer_to_products.values() if len(products) > 1]
print(len(product_lists), "customers reviewed more than one product")

# window=20: every other product in the customer's list is a neighbour, as in a shopping basket
product_model = Word2Vec(product_lists, sg=1, window=20, negative=5, vector_size=50, epochs=30,
                         min_count=3, workers=1, seed=1)

# The product names, so we can print a name instead of its code
with gzip.open(DATA + "products.csv.gz", "rt", encoding="utf-8") as file:
    code_to_title = {row["parent_asin"]: row["title"][:75] for row in csv.DictReader(file)}   # product code -> its name

reviews_per_product = Counter(review["parent_asin"] for review in reviews)   # product code -> how many reviews it has

def find(name):
    # the code of the most-reviewed product whose name contains `name`
    for product, _ in reviews_per_product.most_common():   # (code, count) tuples, most-reviewed first
        if name.lower() in code_to_title[product].lower():
            return product

# For each of four products: its name, then the 3 products with the closest vectors
for name in ["Aeropress Original", "Nespresso", "MWF Refrigerator Water Filter", "Milk Frother"]:
    product = find(name)
    print()
    print(code_to_title[product])
    for similar_product, score in product_model.wv.most_similar(product, topn=3):
        print(f"    {score:.2f}  {code_to_title[similar_product]}")   # the cosine, then the product's name

> 📖 **Reading the output.** The Aeropress coffee maker leads to reusable metal filters made for the Aeropress. A kit
> for refilling Nespresso capsules leads to the paper filters, foil seals and caps that go with it. A GE fridge water
> filter leads to other filters that fit the same fridges. A milk frothing pitcher leads to another frothing pitcher
> and to pour-over coffee filters. The model never read a product name: it only saw which products the same customers
> reviewed.

## Recap

| idea | what you saw |
|---|---|
| Neighbours | the word just before and just after: *"iced latte please"* gives 4 pairs |
| Neighbour counts | a word's vector: one count per vocabulary word; *iced* and *cold* get cosine 1 |
| The catch | one number per vocabulary word: 50,000 numbers, almost all 0, for a real vocabulary |
| The guessing game | real pairs from the text against fake pairs; guess = sigmoid(dot product) |
| The nudge | error = target − guess; each list moves by rate × error × the other list |
| Real reviews | the same nudge in gensim: *fridge* → *refrigerator*, *frig*; *gift* → *birthday*, *christmas* |
| Its limit | *hot* and *cold*, *happy* and *unhappy* come out close: used alike, opposite in meaning |
| Products | customers' product lists as sentences: the Aeropress leads to Aeropress filters |